# ACE-RAG Stage 2 Autonomous Kaggle Run

This notebook starts from the current thesis state: retrieval experiments are done, and Qwen generation needs better evidence packing. It runs the next full Stage-2 comparison automatically.

Kaggle settings: GPU `T4 x2`, Internet `On`.

In [ ]:
import os, shutil, zipfile
from pathlib import Path

input_root = Path('/kaggle/input')
work_root = Path('/kaggle/working')
dst = work_root / 'ace_rag_research'

if dst.exists():
    shutil.rmtree(dst)

folder_candidates = [
    p for p in input_root.glob('**/ace_rag_research')
    if (p / 'ace_rag').exists() and (p / 'experiments').exists()
]

if folder_candidates:
    src = folder_candidates[0]
    print('Found project folder:', src)
    shutil.copytree(src, dst)
else:
    zip_candidates = list(input_root.glob('**/*.zip'))
    print('Zip candidates:', zip_candidates)
    if not zip_candidates:
        raise FileNotFoundError('No ace_rag_research folder or zip found under /kaggle/input')
    zip_path = zip_candidates[0]
    temp = work_root / '_ace_extract'
    if temp.exists():
        shutil.rmtree(temp)
    temp.mkdir(parents=True, exist_ok=True)
    print('Extracting:', zip_path)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(temp)
    extracted_candidates = [
        p for p in temp.glob('**/ace_rag_research')
        if (p / 'ace_rag').exists() and (p / 'experiments').exists()
    ]
    if extracted_candidates:
        shutil.copytree(extracted_candidates[0], dst)
    elif (temp / 'ace_rag').exists() and (temp / 'experiments').exists():
        shutil.copytree(temp, dst)
    else:
        raise FileNotFoundError('Zip extracted, but project folder was not found')

os.chdir(dst)
print('Now in:', Path.cwd())
!ls

## Install And Check Runtime

In [ ]:
!python -m pip install -q -r requirements-cloud.txt
!python -m scripts.cloud_check
!nvidia-smi

## Run Full Stage-2 Autonomous Pipeline

This retrieves chunk and ACE once, loads Qwen once, then evaluates:

- chunk full sources
- chunk packed snippets control
- ACE full sources
- ACE packed snippets at several budgets

It writes metrics after each policy, so partial results are preserved if the session stops.

In [ ]:
!python -m experiments.run_stage2_autonomous \
  --dataset hotpotqa \
  --limit 200 \
  --embed-device cuda \
  --compressor truncate \
  --compress-dims 320 \
  --top-k-nodes 48 \
  --max-expanded-docs 5 \
  --reader-model Qwen/Qwen2.5-1.5B-Instruct \
  --reader-device cuda \
  --reader-batch-size 4 \
  --packed-budgets 160,220,280,340 \
  --focused-budgets 220,280 \
  --snippet-window 1 \
  --max-snippets 8 \
  --max-snippet-tokens 80 \
  --out-dir cloud_results

## Print Summary

In [ ]:
from pathlib import Path
import pandas as pd

paths = sorted(Path('cloud_results').glob('*stage2_autonomous*metrics.csv'))
print(paths)
for path in paths:
    print('\n---', path.name, '---')
    df = pd.read_csv(path)
    display(df)
    cols = ['policy', 'method', 'reader_context', 'packed_token_budget', 'qwen_em', 'qwen_f1', 'avg_reader_evidence_tokens', 'recall@5', 'all_gold@5']
    display(df[[c for c in cols if c in df.columns]].sort_values('qwen_f1', ascending=False))

## Zip Results

In [ ]:
!zip -r ace_rag_stage2_results.zip cloud_results
!ls -lh ace_rag_stage2_results.zip